In [8]:
!pip install pyspark -q
!pip install boto3 -q
!pip install pyarrow -q
!pip install databricks-sql-connector pandas -q

In [9]:
from google.colab import drive
import os

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
    print("✅ Drive mounted successfully")
else:
    print("✅ Drive already mounted, skipping...")

✅ Drive already mounted, skipping...


In [10]:
#configuration notebook

import os
os.chdir("/content/drive/My Drive/Colab Notebooks")
%run oo_config.ipynb

In [11]:
import boto3
import os
import glob
import shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# ── boto3 Client ──────────────────────────────────────────────────────────────
s3 = boto3.client(
    "s3",
    endpoint_url=G_MINIO_ENDPOINT,
    aws_access_key_id=G_MINIO_ACCESS_KEY,
    aws_secret_access_key=G_MINIO_SECRET_KEY
)

# ── Download All Partitioned Files from MinIO ─────────────────────────────────
local_dir  = "/tmp/earthquake"
bucket     = "rawdatasets"
prefix     = "earthquake/"

os.makedirs(local_dir, exist_ok=True)

response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)

for obj in response.get("Contents", []):
    s3_key = obj["Key"]

    # Skip if the key is just the folder prefix itself
    if s3_key.endswith("/"):
        continue

    # Reconstruct local path preserving year=/month= structure
    relative_path = os.path.relpath(s3_key, prefix)
    local_file_path = os.path.join(local_dir, relative_path)

    # Create subdirectories if needed (e.g. year=2023/month=6/)
    os.makedirs(os.path.dirname(local_file_path), exist_ok=True)

    s3.download_file(Bucket=bucket, Key=s3_key, Filename=local_file_path)
    #print(f"Downloaded: {s3_key} → {local_file_path}")

# ── Read All Partitions into Spark DataFrame ──────────────────────────────────
spark = SparkSession.builder \
    .appName("ReadEarthquakeParquet") \
    .getOrCreate()

df = spark.read.parquet(local_dir)

df.printSchema()
#df.show(10, truncate=False)



root
 |-- depth: double (nullable = true)
 |-- lat: double (nullable = true)
 |-- lat_vector: string (nullable = true)
 |-- localdatetime: string (nullable = true)
 |-- location: string (nullable = true)
 |-- location_original: string (nullable = true)
 |-- lon: double (nullable = true)
 |-- lon_vector: string (nullable = true)
 |-- magdefault: double (nullable = true)
 |-- magtypedefault: string (nullable = true)
 |-- n_distancemas: string (nullable = true)
 |-- n_distancerest: string (nullable = true)
 |-- nbm_distancemas: string (nullable = true)
 |-- nbm_distancerest: string (nullable = true)
 |-- status: string (nullable = true)
 |-- utcdatetime: string (nullable = true)
 |-- visible: boolean (nullable = true)
 |-- utcdatetime_ts: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)



In [12]:
# ─── Databricks Connection Config ───────────────────────────────────────────
SERVER_HOSTNAME = G_SERVER_HOSTNAME
HTTP_PATH       = G_HTTP_PATH   # from Connection Details tab
ACCESS_TOKEN    = G_ACCESS_TOKEN         # replace with your PAT

# ─── Catalog / Schema / Table Target ────────────────────────────────────────
CATALOG   = "workspace"            # change to your catalog name (Unity Catalog)
SCHEMA    = "default"         # change to your schema/database
TABLE     = "earthquake_asia"
FULL_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE}"

In [13]:
from databricks import sql

def run_query(cursor, query):
    """Helper to execute and optionally fetch results."""
    cursor.execute(query)
    try:
        return cursor.fetchall()
    except Exception:
        return None

COLUMNS = [
    "depth", "lat", "lat_vector", "localdatetime", "location",
    "location_original", "lon", "lon_vector", "magdefault", "magtypedefault",
    "n_distancemas", "n_distancerest", "nbm_distancemas", "nbm_distancerest",
    "status", "utcdatetime", "visible", "utcdatetime_ts", "year", "month"
]

placeholders = ", ".join(["?"] * len(COLUMNS))
col_names    = ", ".join(COLUMNS)

with sql.connect(
    server_hostname = SERVER_HOSTNAME,
    http_path       = HTTP_PATH,
    access_token    = ACCESS_TOKEN
) as conn:
    with conn.cursor() as cursor:

        # ── Step 1: Set catalog and schema context ───────────────────────────
        cursor.execute(f"USE CATALOG {CATALOG}")
        cursor.execute(f"USE SCHEMA {SCHEMA}")

        # ── Step 2: Create table if not exists ───────────────────────────────
        create_sql = f"""
        CREATE TABLE IF NOT EXISTS {FULL_TABLE} (
            depth            DOUBLE,
            lat              DOUBLE,
            lat_vector       STRING,
            localdatetime    STRING,
            location         STRING,
            location_original STRING,
            lon              DOUBLE,
            lon_vector       STRING,
            magdefault       DOUBLE,
            magtypedefault   STRING,
            n_distancemas    STRING,
            n_distancerest   STRING,
            nbm_distancemas  STRING,
            nbm_distancerest STRING,
            status           STRING,
            utcdatetime      STRING,
            visible          BOOLEAN,
            utcdatetime_ts   TIMESTAMP,
            year  INT,
            month INT
        )
        """
        cursor.execute(create_sql)
        print(f"✅ Table '{FULL_TABLE}' created/verified.")

        # ── Step 3: Insert rows from Pandas DataFrame ─────────────────────────
        #for _, row in pdf.iterrows():
        #    insert_sql = f"""
        #    INSERT INTO {FULL_TABLE} (id, name, score, is_active)
        #    VALUES ({row['id']}, '{row['name']}', {row['score']}, {str(row['is_active']).upper()})
        #    """
        #    cursor.execute(insert_sql)

        # Spark equivalent of: [tuple(r) for r in df.itertuples(index=False)]
        rows = [tuple(row) for row in df.collect()]

        #cursor.executemany(
        #    f"INSERT INTO {FULL_TABLE} VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        #    rows
        #)

        insert_sql = f"""
        INSERT INTO {FULL_TABLE} ({col_names})
        SELECT {placeholders}
        WHERE NOT EXISTS (
            SELECT 1 FROM {FULL_TABLE}
            WHERE utcdatetime        = ?
              AND location           = ?
              AND location_original  = ?
        )
        """


        # Append the 3 key values at end of each row for the WHERE NOT EXISTS check
        rows_with_keys = [
            row + (row[COLUMNS.index("utcdatetime")],
                  row[COLUMNS.index("location")],
                  row[COLUMNS.index("location_original")])
            for row in rows
        ]

        cursor.executemany(insert_sql, rows_with_keys)

        print(f"✅ Inserted {df.count()} rows into '{FULL_TABLE}'.")

        # ── Step 4: Verify by reading back ───────────────────────────────────
        results = run_query(cursor, f"SELECT * FROM {FULL_TABLE}")
        print(f"\n📦 Data in '{FULL_TABLE}':")
        for r in results:
            print(r)

✅ Table 'workspace.default.earthquake_asia' created/verified.
✅ Inserted 698 rows into 'workspace.default.earthquake_asia'.

📦 Data in 'workspace.default.earthquake_asia':
Row(depth=181.0, lat=52.5208740234375, lat_vector='52.5209° U', localdatetime='2023-10-16T19:35:31', location='Kepulauan Andreanof, Kepulauan Aleutian ', location_original='Andreanof Islands, Aleutian Islands', lon=-176.8115692138672, lon_vector='176.8116° B', magdefault=6.400000095367432, magtypedefault='mb', n_distancemas='7,823km NE Pitas,Sabah', n_distancerest='1,642km E Petropavlovsk Kamchatskiy,Russia', nbm_distancemas='7,823km TL Pitas,Sabah', nbm_distancerest='1,642km T Petropavlovsk Kamchatskiy,Russia', status='NORMAL', utcdatetime='2023-10-16T11:35:31', visible=True, utcdatetime_ts=datetime.datetime(2023, 10, 16, 11, 35, 31, tzinfo=<StaticTzInfo 'Etc/UTC'>), year=2023, month=10)
Row(depth=10.0, lat=52.5493278503418, lat_vector='52.5493° U', localdatetime='2026-02-23T13:11:50', location='Fox Islands, Aleutia

In [14]:
# ── Cleanup ───────────────────────────────────────────────────────────────────
shutil.rmtree(local_dir, ignore_errors=True)
print(f"Deleted temp directory: {local_dir}")

Deleted temp directory: /tmp/earthquake
